# Feature Engineering — Technical Indicators

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load & Compute Features

In [ ]:
import pandas as pd

# Load pre-generated feature CSVs (produced by 01_data_exploration.ipynb pipeline cell)
train_feat = pd.read_csv('../data/features/train.csv', index_col='open_time', parse_dates=True)
val_feat   = pd.read_csv('../data/features/val.csv',   index_col='open_time', parse_dates=True)
test_feat  = pd.read_csv('../data/features/test.csv',  index_col='open_time', parse_dates=True)

# Combined view for full-range plots
combined = pd.concat([train_feat, val_feat, test_feat]).sort_index()

print(f"Train : {len(train_feat)} rows  ({train_feat.index[0].date()} → {train_feat.index[-1].date()})")
print(f"Val   : {len(val_feat)} rows  ({val_feat.index[0].date()} → {val_feat.index[-1].date()})")
print(f"Test  : {len(test_feat)} rows  ({test_feat.index[0].date()} → {test_feat.index[-1].date()})")
print(f"\nColumns: {list(combined.columns)}")
train_feat.tail()

## 2. Price with SMA-10 and SMA-50

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(combined.index, combined['close'],  color='steelblue',  linewidth=1.2, label='Close', alpha=0.8)
ax.plot(combined.index, combined['sma_10'], color='darkorange', linewidth=1.5, label='SMA-10')
ax.plot(combined.index, combined['sma_50'], color='crimson',    linewidth=1.5, label='SMA-50')

# Mark golden/death crosses
sma10 = combined['sma_10']
sma50 = combined['sma_50']
cross_up   = (sma10 > sma50) & (sma10.shift(1) <= sma50.shift(1))
cross_down = (sma10 < sma50) & (sma10.shift(1) >= sma50.shift(1))

ax.scatter(combined.index[cross_up],   combined['close'][cross_up],   marker='^', color='seagreen', s=120, zorder=5, label='Golden cross ↑')
ax.scatter(combined.index[cross_down], combined['close'][cross_down], marker='v', color='crimson',  s=120, zorder=5, label='Death cross ↓')

ax.set_title('BTC/USDT — Close Price with SMA-10 and SMA-50', fontsize=14)
ax.set_ylabel('Price (USDT)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/02_sma_crossovers.png', dpi=150)
plt.show()

## 3. RSI — Momentum Oscillator

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Price
ax1.plot(combined.index, combined['close'], color='steelblue', linewidth=1.5)
ax1.set_ylabel('Close Price (USDT)')
ax1.set_title('BTC/USDT — Price and RSI-14', fontsize=14)

# RSI
ax2.plot(combined.index, combined['rsi'], color='purple', linewidth=1.5)
ax2.axhline(70, color='crimson',   linestyle='--', linewidth=1, label='Overbought (70)')
ax2.axhline(30, color='seagreen',  linestyle='--', linewidth=1, label='Oversold (30)')
ax2.axhline(50, color='gray',      linestyle=':',  linewidth=0.8)
ax2.fill_between(combined.index, combined['rsi'], 70,
                 where=(combined['rsi'] >= 70), alpha=0.2, color='crimson')
ax2.fill_between(combined.index, combined['rsi'], 30,
                 where=(combined['rsi'] <= 30), alpha=0.2, color='seagreen')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI')
ax2.legend(loc='upper left')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('../results/figures/02_rsi.png', dpi=150)
plt.show()

## 4. Price Momentum (5-day)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(combined.index, combined['close'], color='steelblue', linewidth=1.5)
ax1.set_ylabel('Close Price (USDT)')
ax1.set_title('BTC/USDT — Price and 5-Day Momentum', fontsize=14)

mom = combined['momentum_5'].dropna()
ax2.bar(mom.index, mom, color=['seagreen' if v >= 0 else 'crimson' for v in mom], width=0.8, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Momentum (%)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('../results/figures/02_momentum.png', dpi=150)
plt.show()

## 5. Feature Correlation Matrix

In [ ]:
feat_cols = ['close', 'sma_10', 'sma_50', 'rsi', 'momentum_5']
corr = train_feat[feat_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix (Training Set)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/02_correlation.png', dpi=150)
plt.show()